In [1]:
import joblib

pipeline = joblib.load('model/fake_news_pipeline.pkl')

In [5]:
model = pipeline['model']
tfidf = pipeline['tfidf']

In [6]:
import re, string
import pandas as pd
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

In [7]:
STOPWORDS = set(stopwords.words('english'))

def simple_clean(text):
    if pd.isna(text):
        return ""
    s = str(text)
    s = re.sub(r"http\S+|www\.\S+|https\S+", " ", s)
    s = re.sub(r"[%s]" % re.escape(string.punctuation), " ", s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    tokens = [w for w in s.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(tokens)

In [8]:
import numpy as np

In [11]:
def explain_text(text,topk=8):
    cleaned = simple_clean(text)
    X = tfidf.transform([cleaned])

    probs = model.predict_proba(X)[0]
    pred = int(model.predict(X)[0])

    coefs = model.coef_[0]

    try:
        feature_names = tfidf.get_feature_names_out()
    except:
        feature_names = tfidf.get_feature_names()

    x_arr = X.toarray()[0]
    contributions = coefs * x_arr

    pos_idx = np.argsort(contributions)[-topk:][::-1]
    neg_idx = np.argsort(contributions)[:topk]

    top_pos = [(feature_names[i], float(contributions[i])) for i in pos_idx if x_arr[i] > 0]
    top_neg = [(feature_names[i], float(contributions[i])) for i in neg_idx if x_arr[i] > 0]

    return {
        "pred": pred,
        "probs": probs.tolist(),
        "top_pos": top_pos,
        "top_neg": top_neg
    }

In [12]:
text = "Breaking: You won't believe what happened next!"
analysis = explain_text(text)
analysis

{'pred': 0,
 'probs': [0.9807380852164916, 0.01926191478350837],
 'top_pos': [],
 'top_neg': [('happened', -0.8323849168641989),
  ('believe', -0.7815279127365805),
  ('breaking', -0.28231481195163044)]}